In [23]:
# 0. imports and paths

import os
import gzip
import tarfile
import io
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, chi2_contingency, kruskal, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

base        = 'D:/TNBC_SV_DNA_Repair'
data_dir    = os.path.join(base, 'dataset')
tables_dir  = os.path.join(base, 'results', 'tables')
figures_dir = os.path.join(base, 'results', 'figures')
scores_dir  = os.path.join(base, 'results', 'scores')

hr_genes      = ['BRCA1','BRCA2','PALB2','RAD51','RAD51B','RAD51C','RAD51D','BRIP1','ATM','CHEK2']
cohesin_genes = ['STAG2','STAG3','SMC1A','SMC1B','RAD21','REC8']
meiosis_genes = ['HORMAD1','HORMAD2','SYCP2','SYCP3','MLH3','MSH4','MSH5']
all_panel     = hr_genes + cohesin_genes + meiosis_genes

group_order  = ['Low', 'Moderate', 'High']
group_colors = {'Low': '#4878cf', 'Moderate': '#f0a500', 'High': '#d94f3d'}

# Load TCGA-BRCA TNBC results for cross-cohort comparisons
gips_df    = pd.read_csv(os.path.join(scores_dir, 'gips_scores.csv'))
gene_score = pd.read_csv(os.path.join(scores_dir, 'gene_disruption_scores.csv'), index_col=0)
tcga_gene_mean = gene_score.mean(axis=1).to_dict()

print('paths set')
print('TCGA-BRCA gene mean disruption loaded:', len(tcga_gene_mean), 'genes')

paths set
TCGA-BRCA gene mean disruption loaded: 23 genes


In [24]:
# 1. Section 1: GSE25066 treatment response — file check + auto-download

import urllib.request

gse_file = os.path.join(data_dir, 'GSE25066_series_matrix.txt.gz')
gse_raw  = os.path.join(data_dir, 'GSE25066_RAW.tar')

if not os.path.exists(gse_file):
    if os.path.exists(gse_raw):
        print(f'Found GSE25066_RAW.tar ({round(os.path.getsize(gse_raw)/1e9, 1)} GB).')
        print('This contains Affymetrix CEL files — not directly usable as a expression matrix.')
        print('Need: GSE25066_series_matrix.txt.gz (processed expression + clinical metadata).')
    print('Attempting auto-download from NCBI FTP...')
    gse_ftp_url = ('https://ftp.ncbi.nlm.nih.gov/geo/series/GSE25nnn/'
                   'GSE25066/matrix/GSE25066_series_matrix.txt.gz')
    try:
        urllib.request.urlretrieve(gse_ftp_url, gse_file)
        print(f'Downloaded: {round(os.path.getsize(gse_file)/1e6, 1)} MB')
    except Exception as ex:
        print(f'Auto-download failed: {ex}')
        print('Manual download: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE25066')

gse_available = os.path.exists(gse_file)
if gse_available:
    print(f'GSE25066 series matrix ready ({round(os.path.getsize(gse_file)/1e6, 1)} MB)')
else:
    print('GSE25066 series matrix not available — Section 1 will be skipped')

GSE25066 series matrix ready (61.6 MB)


In [25]:
# 2. Parse GSE25066 series matrix header (if file available)

if gse_available:
    metadata_lines = {}
    sample_chars   = {}
    matrix_started = False
    sample_ids     = []
    expr_rows      = []
    probe_ids      = []

    with gzip.open(gse_file, 'rt', encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('!series_matrix_table_begin'):
                # next line is the header with sample IDs
                header_line = next(f).rstrip('\n')
                sample_ids  = header_line.split('\t')[1:]
                matrix_started = True
                continue
            if line.startswith('!series_matrix_table_end'):
                break
            if matrix_started:
                parts = line.split('\t')
                probe_ids.append(parts[0].strip('"'))
                expr_rows.append([float(v) if v.strip('"') != '' else np.nan
                                  for v in parts[1:]])
            elif line.startswith('!Sample_geo_accession'):
                metadata_lines['sample_accession'] = line.split('\t')[1:]
            elif line.startswith('!Sample_characteristics_ch1'):
                chars = line.split('\t')[1:]
                # determine characteristic type from first cell
                first = chars[0].strip('"') if chars else ''
                key_raw = first.split(':')[0].strip().lower().replace(' ','_') if ':' in first else f'char_{len(sample_chars)}'
                sample_chars[key_raw] = [c.strip('"').split(':')[-1].strip() for c in chars]

    print(f'Samples: {len(sample_ids)}')
    print(f'Probes: {len(probe_ids)}')
    print(f'Characteristic keys: {list(sample_chars.keys())}')
else:
    print('Skipping — file not available')

Samples: 508
Probes: 22283
Characteristic keys: ['sample_id', 'source', 'age_years', 'er_status_ihc', 'pr_status_ihc', 'her2_status', 'er_status_ihc_esr1_for_indeterminate', 'clinical_t_stage', 'clinical_nodal_status', 'clinical_ajcc_stage', 'grade', 'pathologic_response_pcr_rd', 'pathologic_response_rcb_class', 'drfs_1_event_0_censored', 'drfs_even_time_years', 'esr1_status', 'erbb2_status', 'set_class', 'chemosensitivity_prediction', 'ggi_class', 'pam50_class', 'dlda30_prediction', 'rcb_0_i_prediction', 'tissue']


In [26]:
# 3. Build GSE25066 clinical dataframe and identify TNBC samples

if gse_available and len(sample_ids) > 0:
    clin_df = pd.DataFrame(sample_chars, index=sample_ids)
    print('Clinical dataframe shape:', clin_df.shape)
    print('Columns:', list(clin_df.columns))
    print(clin_df.head(3))

    # Identify ER, PR, HER2 and pCR columns
    er_col   = next((c for c in clin_df.columns if 'er_status_ihc' == c or ('er' in c and 'status' in c and 'esr1' not in c and 'erbb2' not in c)), None)
    pr_col   = next((c for c in clin_df.columns if 'pr_status_ihc' == c or ('pr' in c and 'status' in c)), None)
    her2_col = next((c for c in clin_df.columns if 'her2_status' == c or 'her2' in c), None)
    pcr_col  = next((c for c in clin_df.columns if 'pathologic_response_pcr' in c or 'pcr' in c), None)

    print(f'ER col: {er_col}, PR col: {pr_col}, HER2 col: {her2_col}, pCR col: {pcr_col}')

    # Identify TNBC: all three negative
    # Handles: 'N', 'Negative', 'negative', 'neg', '0', 'NEG'
    def is_negative(s):
        if not isinstance(s, str): return False
        s = s.strip().lower()
        return s in ('n', 'neg', 'negative', '0') or s.startswith('neg')

    tnbc_mask = pd.Series(True, index=clin_df.index)
    for col in [er_col, pr_col, her2_col]:
        if col and col in clin_df.columns:
            tnbc_mask = tnbc_mask & clin_df[col].apply(is_negative)

    gse_tnbc = clin_df[tnbc_mask].index.tolist()
    print(f'TNBC samples in GSE25066: {len(gse_tnbc)}')
else:
    gse_tnbc = []

Clinical dataframe shape: (508, 24)
Columns: ['sample_id', 'source', 'age_years', 'er_status_ihc', 'pr_status_ihc', 'her2_status', 'er_status_ihc_esr1_for_indeterminate', 'clinical_t_stage', 'clinical_nodal_status', 'clinical_ajcc_stage', 'grade', 'pathologic_response_pcr_rd', 'pathologic_response_rcb_class', 'drfs_1_event_0_censored', 'drfs_even_time_years', 'esr1_status', 'erbb2_status', 'set_class', 'chemosensitivity_prediction', 'ggi_class', 'pam50_class', 'dlda30_prediction', 'rcb_0_i_prediction', 'tissue']
            sample_id source age_years er_status_ihc pr_status_ihc  \
"GSM615096"      1002   ISPY      37.8             P             P   
"GSM615097"      1005   ISPY      45.8             P             P   
"GSM615098"      1009   ISPY      40.7             N             N   

            her2_status er_status_ihc_esr1_for_indeterminate clinical_t_stage  \
"GSM615096"           N                                    P               T2   
"GSM615097"           N                

In [27]:
# 4. Build GSE25066 expression matrix and map GPL570 probe IDs to panel genes

if gse_available and len(gse_tnbc) > 0:
    expr_mat = pd.DataFrame(expr_rows, index=probe_ids, columns=sample_ids)
    print('Full expression matrix:', expr_mat.shape)
    print('Sample probe IDs:', list(expr_mat.index[:5]))

    # Check whether index already contains gene symbols
    symbol_check = [p for p in expr_mat.index[:50] if p in all_panel]
    print(f'Panel genes in first 50 index rows: {len(symbol_check)}')

    if len(symbol_check) >= 3:
        panel_in_gse = [g for g in all_panel if g in expr_mat.index]
        gse_expr_panel = expr_mat.loc[panel_in_gse]
        print(f'Panel genes found as symbols: {len(panel_in_gse)}')
    else:
        # GPL570 (Affymetrix HG-U133 Plus 2.0) probe-to-gene mapping for our 23 panel genes
        GPL570_PROBE_MAP = {
            'BRCA1':   ['204531_s_at', '211851_s_at'],
            'BRCA2':   ['208368_s_at'],
            'PALB2':   ['219185_s_at'],
            'RAD51':   ['205024_at', '205748_s_at'],
            'RAD51B':  ['206844_at'],
            'RAD51C':  ['208303_s_at'],
            'RAD51D':  ['204416_at'],
            'BRIP1':   ['219478_at'],
            'ATM':     ['209605_at', '215708_s_at'],
            'CHEK2':   ['205394_at'],
            'STAG2':   ['203397_s_at'],
            'STAG3':   ['228942_at'],
            'SMC1A':   ['201292_at'],
            'SMC1B':   ['228834_at'],
            'RAD21':   ['209636_s_at'],
            'REC8':    ['220007_at'],
            'HORMAD1': ['219748_at'],
            'HORMAD2': ['232430_at'],
            'SYCP2':   ['226831_at'],
            'SYCP3':   ['227869_at'],
            'MLH3':    ['218483_at'],
            'MSH4':    ['210225_s_at'],
            'MSH5':    ['216877_s_at'],
        }

        panel_in_gse = []
        gse_gene_rows = {}
        for gene, probes in GPL570_PROBE_MAP.items():
            found = [p for p in probes if p in expr_mat.index]
            if found:
                gse_gene_rows[gene] = expr_mat.loc[found].mean(axis=0)
                panel_in_gse.append(gene)

        missing = [g for g in all_panel if g not in panel_in_gse]
        print(f'Panel genes mapped via GPL570 probe IDs: {len(panel_in_gse)}/23')
        print(f'Found: {panel_in_gse}')
        print(f'Missing: {missing}')

        gse_expr_panel = pd.DataFrame(gse_gene_rows).T if panel_in_gse else pd.DataFrame()

    print(f'GSE25066 panel expression shape: {gse_expr_panel.shape}')
else:
    panel_in_gse  = []
    gse_expr_panel = pd.DataFrame()
    print('Skipping — TNBC samples not identified or file unavailable')

Full expression matrix: (22283, 508)
Sample probe IDs: ['1007_s_at', '1053_at', '117_at', '121_at', '1255_g_at']
Panel genes in first 50 index rows: 0
Panel genes mapped via GPL570 probe IDs: 12/23
Found: ['BRCA1', 'BRCA2', 'RAD51', 'RAD51B', 'RAD51C', 'BRIP1', 'ATM', 'CHEK2', 'STAG2', 'SMC1A', 'REC8', 'HORMAD1']
Missing: ['PALB2', 'RAD51D', 'STAG3', 'SMC1B', 'RAD21', 'HORMAD2', 'SYCP2', 'SYCP3', 'MLH3', 'MSH4', 'MSH5']
GSE25066 panel expression shape: (12, 508)


In [28]:
# 5. Compute GIPS for GSE25066 TNBC (expression-only)

if gse_available and len(gse_tnbc) > 0 and len(panel_in_gse) >= 5:
    # Subset to TNBC samples; gse_expr_panel columns = all sample_ids
    tnbc_in_panel = [s for s in gse_tnbc if s in gse_expr_panel.columns]
    gse_expr_tnbc = gse_expr_panel[tnbc_in_panel]

    # Log-transform if raw intensity scale (typical Affymetrix values > 50)
    if gse_expr_tnbc.values.max() > 50:
        gse_expr_tnbc = np.log2(gse_expr_tnbc + 1)
        print('Log2 transformed GPL570 intensities')
    else:
        print('Expression already in log scale')

    # Expression z-score disruption indicator
    gse_z    = gse_expr_tnbc.apply(lambda row: (row - row.mean()) / (row.std() + 1e-8), axis=1)
    gse_disr = (gse_z.abs() > 1.96).astype(float)

    # GIPS (expression modality only — no CN/mutation in GSE25066)
    gse_gips_raw    = gse_disr.sum(axis=0)
    gse_gips_scaled = (gse_gips_raw - gse_gips_raw.min()) / (gse_gips_raw.max() - gse_gips_raw.min())
    t33_g = gse_gips_scaled.quantile(0.333)
    t66_g = gse_gips_scaled.quantile(0.667)

    def gse_group(x):
        if x <= t33_g:   return 'Low'
        elif x <= t66_g: return 'Moderate'
        return 'High'

    gse_gips_df = pd.DataFrame({
        'sample':      tnbc_in_panel,
        'GIPS_scaled': gse_gips_scaled[tnbc_in_panel].values,
        'GIPS_group':  gse_gips_scaled[tnbc_in_panel].apply(gse_group).values,
    })
    print(f'GSE25066 TNBC cohort: {len(tnbc_in_panel)} samples, {len(panel_in_gse)} genes')
    print('GIPS groups:', gse_gips_df['GIPS_group'].value_counts().to_dict())
else:
    gse_gips_df = None
    if not gse_available:
        print('No GSE25066 analysis performed (file not found)')
    elif len(gse_tnbc) == 0:
        print('No TNBC samples identified in GSE25066')
    else:
        print(f'Only {len(panel_in_gse)} panel genes found — insufficient for GIPS (need ≥5)')

Expression already in log scale
GSE25066 TNBC cohort: 178 samples, 12 genes
GIPS groups: {'Low': 105, 'Moderate': 49, 'High': 24}


In [30]:
# 6. Chi-square test GIPS vs pCR (GSE25066)

if gse_available and gse_gips_df is not None and pcr_col and pcr_col in clin_df.columns:
    gse_merged = gse_gips_df.merge(
        clin_df[[pcr_col]].rename(columns={pcr_col: 'pcr'}).reset_index().rename(columns={'index':'sample'}),
        on='sample', how='inner'
    )

    # Print raw values first to inspect
    print('Raw pCR column unique values:', gse_merged['pcr'].unique())

    def is_pcr(s):
        if not isinstance(s, str): return np.nan
        s = s.strip().lower()
        if s in ('pcr', 'pcr', 'complete response', '1', 'yes', 'cr'):
            return 1
        if s in ('rd', 'no pcr', 'residual disease', '0', 'no', 'not pcr'):
            return 0
        # Handle pCR/RD format
        if s.startswith('pcr'):
            return 1
        if s.startswith('rd'):
            return 0
        return np.nan

    gse_merged['pcr_bin'] = gse_merged['pcr'].apply(is_pcr)

    print('Before dropping NaN:', gse_merged['pcr_bin'].value_counts(dropna=False).to_dict())

    gse_merged = gse_merged.dropna(subset=['pcr_bin'])
    gse_merged['pcr_bin'] = gse_merged['pcr_bin'].astype(int)

    print('GSE25066 merged shape:', gse_merged.shape)
    print('pCR counts:', gse_merged['pcr_bin'].value_counts().to_dict())

    if gse_merged['pcr_bin'].nunique() < 2:
        print('Only one pCR class found — cannot run chi-square. Check raw values above.')
        ct = None
        p_chi = np.nan
    else:
        ct = pd.crosstab(gse_merged['GIPS_group'], gse_merged['pcr_bin'])
        ct.columns = ['no_pCR', 'pCR']
        print('\nObserved vs Expected (3x2 table):')
        print(ct)
        chi2, p_chi, dof, expected = chi2_contingency(ct.values)
        print(f'\nChi-square: chi2={round(chi2,3)}, df={dof}, p={round(p_chi,4)}')
else:
    print('Chi-square test skipped — GSE25066 not available or pCR column not found')
    gse_merged = None
    ct = None
    p_chi = np.nan

Raw pCR column unique values: ['RD' 'pCR' 'NA' 'RCB-III' 'RCB-0/I' 'RCB-II']
Before dropping NaN: {0.0: 75, nan: 66, 1.0: 37}
GSE25066 merged shape: (112, 5)
pCR counts: {0: 75, 1: 37}

Observed vs Expected (3x2 table):
            no_pCR  pCR
GIPS_group             
High             9    4
Low             40   23
Moderate        26   10

Chi-square: chi2=0.823, df=2, p=0.6625


In [31]:
# 7. Logistic regression pCR ~ GIPS (GSE25066)

if gse_merged is not None and len(gse_merged) > 10:
    try:
        import statsmodels.api as sm
        X = sm.add_constant(gse_merged['GIPS_scaled'].values)
        y = gse_merged['pcr_bin'].values
        logit_model = sm.Logit(y, X)
        result = logit_model.fit(disp=0)
        coef    = result.params[1]
        se      = result.bse[1]
        OR      = round(np.exp(coef), 3)
        ci_lo   = round(np.exp(coef - 1.96*se), 3)
        ci_hi   = round(np.exp(coef + 1.96*se), 3)
        p_logit = round(result.pvalues[1], 4)
        print(f'Logistic regression: OR={OR} (95% CI: {ci_lo}-{ci_hi}), p={p_logit}')
    except Exception as ex:
        print(f'Logistic regression error: {ex}')
        OR, ci_lo, ci_hi, p_logit = np.nan, np.nan, np.nan, np.nan
else:
    print('Logistic regression skipped')
    OR, ci_lo, ci_hi, p_logit = np.nan, np.nan, np.nan, np.nan

Logistic regression: OR=0.381 (95% CI: 0.041-3.538), p=0.3959


In [32]:
# 8. pCR rate bar chart by GIPS tertile (GSE25066)

if gse_merged is not None and len(gse_merged) > 5:
    pcr_rates = gse_merged.groupby('GIPS_group')['pcr_bin'].agg(['mean','count'])
    pcr_rates.columns = ['pcr_rate', 'n']
    pcr_rates = pcr_rates.reindex([g for g in group_order if g in pcr_rates.index])

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(
        range(len(pcr_rates)),
        pcr_rates['pcr_rate'] * 100,
        color=[group_colors[g] for g in pcr_rates.index],
        edgecolor='none',
        alpha=0.85
    )
    for i, (idx, row) in enumerate(pcr_rates.iterrows()):
        ax.text(i, row['pcr_rate']*100 + 1, f'n={int(row["n"])}', ha='center', fontsize=9)
    ax.set_xticks(range(len(pcr_rates)))
    ax.set_xticklabels(pcr_rates.index, fontsize=11)
    ax.set_ylabel('pCR rate (%)')
    ax.set_title(f'GSE25066: pCR rate by GIPS tertile (chi2 p={round(p_chi,4)})' if ct is not None else 'GSE25066: pCR by GIPS')
    plt.tight_layout()
    fig.savefig(os.path.join(figures_dir, 'nb6_pcr_gips.png'), dpi=150)
    plt.show()
    print('saved pCR bar chart')

    # Save GSE25066 results
    gse_out = gse_merged[['sample','GIPS_scaled','GIPS_group','pcr_bin']].copy()
    gse_out.to_csv(os.path.join(tables_dir, 'gse25066_pcr_gips.csv'), index=False)
    print('Saved gse25066_pcr_gips.csv')
else:
    print('pCR bar chart skipped — insufficient data')

saved pCR bar chart
Saved gse25066_pcr_gips.csv


In [33]:
# 9. Section 2: TCGA-OV cross-cancer replication
# Load TCGA-OV expression

ov_expr_file = os.path.join(data_dir, 'TCGA.OV.sampleMap_HiSeqV2.gz')
print(f'OV expression file exists: {os.path.exists(ov_expr_file)}')
print(f'File size: {round(os.path.getsize(ov_expr_file)/1e9,2)} GB')

with gzip.open(ov_expr_file, 'rt') as f:
    ov_expr = pd.read_csv(f, sep='\t', index_col=0)

print('OV expression shape:', ov_expr.shape)
print('Index sample (first 5):', list(ov_expr.index[:5]))
print('Sample columns (first 3):', list(ov_expr.columns[:3]))

OV expression file exists: True
File size: 0.02 GB
OV expression shape: (20530, 308)
Index sample (first 5): ['ARHGEF10L', 'HIF3A', 'RNF17', 'RNF10', 'RNF11']
Sample columns (first 3): ['TCGA-61-1910-01', 'TCGA-61-1728-01', 'TCGA-09-1666-01']


In [34]:
# 10. check OV expression gene format and find panel genes

coord_check = [g for g in ov_expr.index[:20] if g.startswith('chr') or ':' in g]
print(f'Coordinate-format rows: {len(coord_check)} / 20')

if len(coord_check) == 0:
    # Gene symbols as index — direct lookup
    panel_in_ov = [g for g in all_panel if g in ov_expr.index]
    print(f'OV uses gene symbols. Panel genes found: {len(panel_in_ov)}')
    print('Found:', panel_in_ov)
    missing_ov = [g for g in all_panel if g not in ov_expr.index]
    print('Missing:', missing_ov)
    ov_gene_format = 'symbols'
else:
    print('OV uses coordinate format — coordinate mapping required (same as BRCA exon file)')
    ov_gene_format = 'coordinates'
    panel_in_ov = []

Coordinate-format rows: 0 / 20
OV uses gene symbols. Panel genes found: 21
Found: ['BRCA1', 'BRCA2', 'PALB2', 'RAD51', 'RAD51C', 'BRIP1', 'ATM', 'CHEK2', 'STAG2', 'STAG3', 'SMC1A', 'SMC1B', 'RAD21', 'REC8', 'HORMAD1', 'HORMAD2', 'SYCP2', 'SYCP3', 'MLH3', 'MSH4', 'MSH5']
Missing: ['RAD51B', 'RAD51D']


In [36]:
# 11. (conditional) coordinate mapping for OV if needed

if ov_gene_format == 'coordinates' and len(panel_in_ov) < 5:
    import mygene
    mg = mygene.MyGeneInfo()
    ov_query = mg.querymany(
        all_panel,
        scopes='symbol',
        fields='symbol,genomic_pos',
        species='human',
        as_dataframe=True
    )
    ov_query = ov_query[~ov_query.index.duplicated(keep='first')]
    ov_query = ov_query.dropna(subset=['genomic_pos.chr'])

    ov_gene_expr_rows = {}
    for gene in all_panel:
        if gene not in ov_query.index:
            continue
        row = ov_query.loc[gene]
        gc, gs, ge = str(row['genomic_pos.chr']), int(row['genomic_pos.start']), int(row['genomic_pos.end'])
        matched = []
        for idx in ov_expr.index:
            try:
                c  = idx.split(':')[0].replace('chr','')
                se = idx.split(':')[1].split('-')
                s, e = int(se[0]), int(se[1])
                if c == gc and not (e < gs or s > ge):
                    matched.append(idx)
            except: continue
        if matched:
            ov_gene_expr_rows[gene] = matched
            print(f'{gene}: {len(matched)} rows')

    # Aggregate
    ov_gene_expr = {}
    for gene, rows in ov_gene_expr_rows.items():
        ov_gene_expr[gene] = ov_expr.loc[rows].mean(axis=0)
    ov_panel_expr = pd.DataFrame(ov_gene_expr).T
    panel_in_ov = list(ov_panel_expr.index)
    print('Panel genes mapped via coordinates:', len(panel_in_ov))
else:
    # Direct symbol access
    ov_panel_expr = ov_expr.loc[panel_in_ov]
    print('Using direct gene symbol access. Shape:', ov_panel_expr.shape)

Using direct gene symbol access. Shape: (21, 308)


In [37]:
# 12. Load TCGA-OV copy number

ov_cn_file = os.path.join(data_dir, 'TCGA.OV.sampleMap_Gistic2_CopyNumber_Gistic2_all_thresholded.by_genes.gz')
print(f'OV CN file exists: {os.path.exists(ov_cn_file)}')

with gzip.open(ov_cn_file, 'rt') as f:
    ov_cn = pd.read_csv(f, sep='\t', index_col=0)

print('OV CN shape:', ov_cn.shape)
print('Index sample:', list(ov_cn.index[:5]))

panel_in_ov_cn = [g for g in all_panel if g in ov_cn.index]
print('Panel genes in OV CN:', len(panel_in_ov_cn))
missing_cn = [g for g in all_panel if g not in ov_cn.index]
print('Missing:', missing_cn)

OV CN file exists: True
OV CN shape: (24776, 579)
Index sample: ['ACAP3', 'ACTRT2', 'AGRN', 'ANKRD65', 'ATAD3A']
Panel genes in OV CN: 23
Missing: []


In [38]:
# 13. Load TCGA-OV clinical data

ov_clin_tsv = os.path.join(data_dir, 'ov_tcga_pub_clinical_data.tsv')
ov_clin_tar = os.path.join(data_dir, 'clinical.project-tcga-ov.2026-05-22.tar.gz')

ov_clin = None

# Try cBioPortal TSV first
if os.path.exists(ov_clin_tsv):
    ov_clin_cb = pd.read_csv(ov_clin_tsv, sep='\t')
    print('cBioPortal OV clinical shape:', ov_clin_cb.shape)
    print('Columns:', list(ov_clin_cb.columns))
    os_cols  = [c for c in ov_clin_cb.columns if 'os' in c.lower() or 'overall' in c.lower()]
    pfi_cols = [c for c in ov_clin_cb.columns if 'pfi' in c.lower() or 'progression' in c.lower() or 'relapse' in c.lower()]
    print('OS candidate cols:', os_cols)
    print('PFI candidate cols:', pfi_cols)

# Try GDC tar
if os.path.exists(ov_clin_tar):
    with tarfile.open(ov_clin_tar, 'r:gz') as tar:
        tsv_members = [m for m in tar.getnames() if m.endswith('.tsv')]
        print('TSV files in GDC tar:', tsv_members)
        if tsv_members:
            clinical_tsv = next((m for m in tsv_members if 'clinical' in m.lower()), tsv_members[0])
            f = tar.extractfile(clinical_tsv)
            ov_gdc = pd.read_csv(io.BytesIO(f.read()), sep='\t')
            print('GDC OV clinical shape:', ov_gdc.shape)
            gdc_os_cols  = [c for c in ov_gdc.columns if 'vital' in c.lower() or 'overall' in c.lower() or 'days_to_death' in c.lower()]
            gdc_pfi_cols = [c for c in ov_gdc.columns if 'progression' in c.lower() or 'recurrence' in c.lower()]
            print('GDC OS cols:', gdc_os_cols[:5])
            print('GDC PFI cols:', gdc_pfi_cols[:5])

cBioPortal OV clinical shape: (489, 25)
Columns: ['Study ID', 'Patient ID', 'Sample ID', 'ACGH Data', 'Cancer Type', 'Cancer Type Detailed', 'Complete Data', 'Disease Free (Months)', 'Disease Free Status', 'Fraction Genome Altered', 'Neoplasm Histologic Grade', 'MRNA Data', 'Mutation Count', 'Oncotree Code', 'Overall Survival (Months)', 'Overall Survival Status', 'Platinum Status', 'Primary Therapy Outcome Success', 'Number of Samples Per Patient', 'Sample Type', 'Sequenced', 'Somatic Status', 'TMB (nonsynonymous)', 'Tumor Residual Disease', 'Tumor Stage 2009']
OS candidate cols: ['Overall Survival (Months)', 'Overall Survival Status']
PFI candidate cols: []
TSV files in GDC tar: ['clinical.tsv', 'family_history.tsv', 'exposure.tsv', 'pathology_detail.tsv', 'follow_up.tsv']
GDC OV clinical shape: (4223, 201)
GDC OS cols: ['demographic.days_to_death', 'demographic.vital_status', 'diagnoses.best_overall_response', 'diagnoses.days_to_best_overall_response']
GDC PFI cols: ['diagnoses.days_

In [39]:
# 14. Build OV clinical survival table

if os.path.exists(ov_clin_tsv):
    ov_clin_cb = pd.read_csv(ov_clin_tsv, sep='\t')

    # Standardize sample ID column
    id_col = next((c for c in ov_clin_cb.columns if 'sample' in c.lower() or 'patient' in c.lower()), ov_clin_cb.columns[0])

    # Direct column name lookup using exact names from cBioPortal output
    os_time_col  = next((c for c in ov_clin_cb.columns if c == 'Overall Survival (Months)'), None)
    os_event_col = next((c for c in ov_clin_cb.columns if c == 'Overall Survival Status'), None)
    pfi_time_col = next((c for c in ov_clin_cb.columns if c == 'Disease Free (Months)'), None)
    pfi_event_col= next((c for c in ov_clin_cb.columns if c == 'Disease Free Status'), None)

    # Fallback: broader search if exact names not found
    if not os_time_col:
        os_time_col = next((c for c in ov_clin_cb.columns if 'survival' in c.lower() and ('month' in c.lower() or 'time' in c.lower())), None)
    if not os_event_col:
        os_event_col = next((c for c in ov_clin_cb.columns if 'survival' in c.lower() and 'status' in c.lower()), None)
    if not pfi_time_col:
        pfi_time_col = next((c for c in ov_clin_cb.columns if ('disease free' in c.lower() or 'progression' in c.lower()) and ('month' in c.lower() or 'time' in c.lower())), None)
    if not pfi_event_col:
        pfi_event_col = next((c for c in ov_clin_cb.columns if ('disease free' in c.lower() or 'progression' in c.lower()) and 'status' in c.lower()), None)

    print(f'ID col: {id_col}')
    print(f'OS time: {os_time_col}, OS event: {os_event_col}')
    print(f'PFI time: {pfi_time_col}, PFI event: {pfi_event_col}')

    surv_cols = [id_col]
    if os_time_col:  surv_cols.append(os_time_col)
    if os_event_col: surv_cols.append(os_event_col)
    if pfi_time_col: surv_cols.append(pfi_time_col)
    if pfi_event_col:surv_cols.append(pfi_event_col)

    ov_surv = ov_clin_cb[surv_cols].copy()
    ov_surv.columns = ['sample'] + surv_cols[1:]

    # Encode OS event: cBioPortal format is '0:LIVING' and '1:DECEASED'
    if os_event_col:
        ov_surv['os_event_bin'] = ov_surv[os_event_col].astype(str).str.contains('1:|DECEASED|Dead', case=False, na=False).astype(int)
        print('OS events:', int(ov_surv['os_event_bin'].sum()))
    else:
        print('OS event column not found')

    if pfi_event_col:
        ov_surv['pfi_event_bin'] = ov_surv[pfi_event_col].astype(str).str.contains('1:|Recurred|Progressed', case=False, na=False).astype(int)
        print('PFI events:', int(ov_surv['pfi_event_bin'].sum()))
    else:
        print('PFI event column not found')

    print('OV survival table shape:', ov_surv.shape)

else:
    print('cBioPortal OV clinical file not found')
    ov_surv = None
    os_time_col = pfi_time_col = None

ID col: Patient ID
OS time: Overall Survival (Months), OS event: Overall Survival Status
PFI time: Disease Free (Months), PFI event: Disease Free Status
OS events: 272
PFI events: 349
OV survival table shape: (489, 7)


In [40]:
# 15. Intersect expression, CN, and survival for OV cohort

ov_expr_samples = set(ov_panel_expr.columns.tolist())
ov_cn_samples   = set(ov_cn.columns.tolist())
ov_surv_samples = set(ov_surv['sample'].tolist()) if ov_surv is not None else set()

# Note: OV sample IDs may differ between expression/CN (TCGA barcode) and clinical (shorter)
# Expression/CN use TCGA-XX-XXXX-01 format; clinical may use TCGA-XX-XXXX
def shorten_tcga_id(s):
    parts = s.split('-')
    return '-'.join(parts[:3]) if len(parts) >= 3 else s

# Try full barcode intersection first
ov_common_full = ov_expr_samples & ov_cn_samples
print(f'Expr samples: {len(ov_expr_samples)}')
print(f'CN samples: {len(ov_cn_samples)}')
print(f'Expr & CN overlap (full IDs): {len(ov_common_full)}')

# Also check survival overlap
if ov_surv is not None:
    ov_surv_full = ov_expr_samples & ov_cn_samples & ov_surv_samples
    print(f'Three-way overlap (full IDs): {len(ov_surv_full)}')

    if len(ov_surv_full) < 10:
        # Try shorter IDs
        expr_short = {shorten_tcga_id(s): s for s in ov_expr_samples}
        cn_short   = {shorten_tcga_id(s): s for s in ov_cn_samples}
        surv_short = {shorten_tcga_id(s): s for s in ov_surv_samples}
        common_short = set(expr_short) & set(cn_short) & set(surv_short)
        print(f'Three-way overlap (shortened IDs): {len(common_short)}')
        ov_common = [expr_short[k] for k in common_short]
        ov_common_surv_ids = [surv_short[k] for k in common_short]
    else:
        ov_common = list(ov_surv_full)
        ov_common_surv_ids = ov_common
else:
    ov_common = list(ov_common_full)
    ov_common_surv_ids = ov_common

print(f'Final OV cohort size: {len(ov_common)}')

Expr samples: 308
CN samples: 579
Expr & CN overlap (full IDs): 301
Three-way overlap (full IDs): 0
Three-way overlap (shortened IDs): 293
Final OV cohort size: 293


In [41]:
# 16. Apply 23-gene panel to OV cohort and compute GIPS

ov_common = [s for s in ov_common if s in ov_panel_expr.columns and s in ov_cn.columns]

# Common genes across panel, expression, and CN
ov_panel_use = [g for g in panel_in_ov if g in panel_in_ov_cn]
print(f'Panel genes usable for OV GIPS: {len(ov_panel_use)}')

ov_expr_sub = ov_panel_expr.loc[ov_panel_use, ov_common]
ov_cn_sub   = ov_cn.loc[ov_panel_use, ov_common]

# Log transform if needed
if ov_expr_sub.values.max() > 50:
    ov_expr_sub = np.log2(ov_expr_sub + 1)
    print('Log2 transformed OV expression')
else:
    print('OV expression already log scale')

# Expression z-score disruption
ov_z    = ov_expr_sub.apply(lambda row: (row - row.mean())/(row.std()+1e-8), axis=1)
ov_expr_disr = (ov_z.abs() > 1.96).astype(float)

# CN disruption
ov_cn_disr = (ov_cn_sub != 0).astype(float)

# 2-modality gene score
ov_gene_score = (ov_expr_disr + ov_cn_disr) / 2.0

# GIPS
ov_gips_raw    = ov_gene_score.sum(axis=0)
ov_gips_scaled = (ov_gips_raw - ov_gips_raw.min()) / (ov_gips_raw.max() - ov_gips_raw.min())
ov_t33 = ov_gips_scaled.quantile(0.333)
ov_t66 = ov_gips_scaled.quantile(0.667)

def ov_group(x):
    if x <= ov_t33: return 'Low'
    elif x <= ov_t66: return 'Moderate'
    return 'High'

ov_gips_df = pd.DataFrame({
    'sample':      ov_common,
    'GIPS_scaled': ov_gips_scaled.values,
    'GIPS_group':  ov_gips_scaled.apply(ov_group).values,
})

print('OV GIPS groups:', ov_gips_df['GIPS_group'].value_counts().to_dict())

Panel genes usable for OV GIPS: 21
OV expression already log scale
OV GIPS groups: {'Low': 109, 'High': 95, 'Moderate': 86}


In [46]:
# 17. OV Kaplan-Meier survival

from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test, logrank_test

if ov_surv is not None:

    # Rebuild the ID map directly from ov_common after cell 16 filtering
    def shorten_tcga_id(s):
        parts = s.split('-')
        return '-'.join(parts[:3]) if len(parts) >= 3 else s

    ov_gips_surv = ov_gips_df.copy()
    ov_gips_surv['surv_id'] = ov_gips_surv['sample'].apply(shorten_tcga_id)

    ov_merged = ov_gips_surv.merge(
        ov_surv.rename(columns={'sample': 'surv_id'}),
        on='surv_id', how='inner'
    )

    print('OV survival+GIPS shape:', ov_merged.shape)
    if 'os_event_bin' in ov_merged.columns:
        print('OS events in merged:', int(ov_merged['os_event_bin'].sum()))
    if 'pfi_event_bin' in ov_merged.columns:
        print('PFI events in merged:', int(ov_merged['pfi_event_bin'].sum()))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, tc, ec, label in [
        (axes[0], 'Overall Survival (Months)',  'os_event_bin',  'OS'),
        (axes[1], 'Disease Free (Months)', 'pfi_event_bin', 'PFI'),
    ]:
        if tc not in ov_merged.columns or ec not in ov_merged.columns:
            ax.set_visible(False)
            continue
        valid = ov_merged.dropna(subset=[tc, ec])
        valid = valid.copy()
        valid[tc] = pd.to_numeric(valid[tc], errors='coerce')
        valid = valid.dropna(subset=[tc])
        kmf = KaplanMeierFitter()
        for grp in group_order:
            sub = valid[valid['GIPS_group']==grp]
            if len(sub) < 3: continue
            kmf.fit(sub[tc], sub[ec], label=f'{grp} (n={len(sub)})')
            kmf.plot_survival_function(ax=ax, ci_show=True, color=group_colors[grp])
        if len(valid) > 5:
            res = multivariate_logrank_test(valid[tc], valid['GIPS_group'], valid[ec])
            lo = valid[valid['GIPS_group']=='Low']
            hi = valid[valid['GIPS_group']=='High']
            if len(lo) > 3 and len(hi) > 3:
                pw = logrank_test(lo[tc], hi[tc], lo[ec], hi[ec])
                pw_str = f', Low vs High p={round(pw.p_value,4)}'
            else:
                pw_str = ''
            ax.set_title(f'TCGA-OV {label} by GIPS (p={round(res.p_value,4)}{pw_str})', fontsize=9)
        ax.set_xlabel('months')
        ax.set_ylabel('survival probability')
        ax.legend(fontsize=8)

    plt.suptitle(f'TCGA-OV survival analysis (n={len(ov_merged)})', fontsize=11)
    plt.tight_layout()
    fig.savefig(os.path.join(figures_dir, 'nb6_ov_km.png'), dpi=150)
    plt.show()
    print('saved OV KM curves')
else:
    ov_merged = ov_gips_df.copy()
    print('No survival data — KM skipped')

OV survival+GIPS shape: (290, 10)
OS events in merged: 164
PFI events in merged: 202
saved OV KM curves


In [47]:
# 18. OV Cox regression

from lifelines import CoxPHFitter

ov_cox_results = []

for tc, ec, label in [
    (os_time_col,  'os_event_bin',  'OS') if ov_surv is not None else (None, None, None),
    (pfi_time_col, 'pfi_event_bin', 'PFI') if ov_surv is not None else (None, None, None),
]:
    if tc is None or 'ov_merged' not in dir() or tc not in ov_merged.columns or ec not in ov_merged.columns:
        continue
    valid = ov_merged[['GIPS_scaled', tc, ec]].dropna()
    valid.columns = ['GIPS', 'duration', 'event']
    valid['duration'] = pd.to_numeric(valid['duration'], errors='coerce')
    valid = valid.dropna()
    if len(valid) < 10:
        print(f'{label}: insufficient data')
        continue
    try:
        cph = CoxPHFitter()
        cph.fit(valid, duration_col='duration', event_col='event')
        s    = cph.summary
        hr   = round(np.exp(s['coef'].values[0]), 3)
        ci_l = round(np.exp(s['coef lower 95%'].values[0]), 3)
        ci_h = round(np.exp(s['coef upper 95%'].values[0]), 3)
        p    = round(s['p'].values[0], 4)
        print(f'{label}: HR={hr} (95% CI: {ci_l}-{ci_h}), p={p}')
        ov_cox_results.append({'endpoint': label, 'HR': hr, 'CI_low': ci_l, 'CI_high': ci_h, 'p': p})
    except Exception as ex:
        print(f'{label}: {ex}')

ov_cox_df = pd.DataFrame(ov_cox_results)
ov_cox_df.to_csv(os.path.join(tables_dir, 'tcga_ov_cox.csv'), index=False)
print(ov_cox_df)

OS: HR=1.244 (95% CI: 0.522-2.965), p=0.622
PFI: HR=1.865 (95% CI: 0.737-4.718), p=0.1881
  endpoint     HR  CI_low  CI_high       p
0       OS  1.244   0.522    2.965  0.6220
1      PFI  1.865   0.737    4.718  0.1881


In [48]:
# 19. Cross-cancer Spearman correlation: TCGA-BRCA TNBC vs TCGA-OV

ov_gene_mean = ov_gene_score.mean(axis=1).to_dict()

common_cancer_genes = sorted(set(tcga_gene_mean.keys()) & set(ov_gene_mean.keys()) & set(all_panel))
print('Common genes for cross-cancer correlation:', len(common_cancer_genes))

brca_vals = np.array([tcga_gene_mean[g] for g in common_cancer_genes])
ov_vals   = np.array([ov_gene_mean[g]   for g in common_cancer_genes])

r_cc, p_cc = spearmanr(brca_vals, ov_vals)
print(f'Cross-cancer Spearman r = {round(r_cc,3)}, p = {round(p_cc,4)}')

# Scatter plot with gene labels and functional group colours
gene_group = {g: 'HR' if g in hr_genes else 'Cohesin' if g in cohesin_genes else 'Meiosis'
              for g in common_cancer_genes}
palette = {'HR': '#d94f3d', 'Cohesin': '#f0a500', 'Meiosis': '#4878cf'}
point_colors = [palette[gene_group[g]] for g in common_cancer_genes]

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(brca_vals, ov_vals, c=point_colors, s=70, alpha=0.85, edgecolors='white', linewidths=0.5)
for i, g in enumerate(common_cancer_genes):
    ax.annotate(g, (brca_vals[i], ov_vals[i]), fontsize=7.5, alpha=0.75,
                xytext=(3,3), textcoords='offset points')

# Trend line
if len(brca_vals) > 2:
    z = np.polyfit(brca_vals, ov_vals, 1)
    p_fit = np.poly1d(z)
    x_line = np.linspace(brca_vals.min(), brca_vals.max(), 100)
    ax.plot(x_line, p_fit(x_line), 'k--', alpha=0.4, linewidth=1)

ax.set_xlabel('TCGA-BRCA TNBC mean disruption score')
ax.set_ylabel('TCGA-OV mean disruption score')
ax.set_title(f'Cross-cancer gene disruption correlation\nSpearman r={round(r_cc,3)}, p={round(p_cc,4)}')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=v, label=k) for k, v in palette.items()], fontsize=9)

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb6_cross_cancer_spearman.png'), dpi=150)
plt.show()
print('saved cross-cancer scatter')

Common genes for cross-cancer correlation: 21
Cross-cancer Spearman r = 0.35, p = 0.1203
saved cross-cancer scatter


In [50]:
# 20. Save OV results and notebook 6 summary

ov_gips_df.to_csv(os.path.join(tables_dir, 'tcga_ov_gips_survival.csv'), index=False)

# Cross-cancer Spearman table
cc_spearman = pd.DataFrame({
    'gene':       common_cancer_genes,
    'BRCA_mean':  brca_vals,
    'OV_mean':    ov_vals,
    'group':      [gene_group[g] for g in common_cancer_genes],
})
cc_spearman.to_csv(os.path.join(tables_dir, 'cross_cancer_spearman.csv'), index=False)

summary6 = {
    'GSE25066 file available':           str(gse_available),
    'OV expression shape':               str(ov_panel_expr.shape),
    'OV panel genes available':          len(ov_panel_use),
    'OV cohort size':                    len(ov_gips_df),
    'OV GIPS groups':                    str(ov_gips_df['GIPS_group'].value_counts().to_dict()),
    'Cross-cancer genes':                len(common_cancer_genes),
    'Cross-cancer Spearman r':           round(r_cc, 3),
    'Cross-cancer Spearman p':           round(p_cc, 4),
}

if len(ov_cox_results) > 0:
    for res in ov_cox_results:
        summary6[f'OV Cox {res["endpoint"]} HR'] = res['HR']
        summary6[f'OV Cox {res["endpoint"]} p']  = res['p']

for k, v in summary6.items():
    print(f'{k}: {v}')

pd.DataFrame.from_dict(summary6, orient='index', columns=['value']).to_csv(
    os.path.join(tables_dir, 'nb6_summary.csv'))
print('notebook 6 complete')

GSE25066 file available: True
OV expression shape: (21, 308)
OV panel genes available: 21
OV cohort size: 290
OV GIPS groups: {'Low': 109, 'High': 95, 'Moderate': 86}
Cross-cancer genes: 21
Cross-cancer Spearman r: 0.35
Cross-cancer Spearman p: 0.1203
OV Cox OS HR: 1.244
OV Cox OS p: 0.622
OV Cox PFI HR: 1.865
OV Cox PFI p: 0.1881
notebook 6 complete
